# parse_instruction_ver13 (multi task)

## 目的
- これまでの primary task 予測から、複数 task (multi task) 予測へ拡張する。
- 1つの正解 task だけでなく、GT の `tasks` 全体との一致度を評価する。

## 方針
- v13a: 既存 single-task モデルをそのまま multi task に流用 (基準線)
- v13b: preserve/stabilize 系を規則追加して 2個目以降の task を補完
- v13c: 類似例の GT task 列を転写して multi task 化
- v13d: v13b/v13c の簡易アンサンブル

## 注意
- grouped データ (`annotations_grouped_ver01/02.json`) は推論評価入力としてのみ使用する。
- モデル規則の設計・重み調整は既知データ上の試行に限定する。

In [1]:
from __future__ import annotations

import copy
import json
import re
import sys
from pathlib import Path
from pprint import pprint

WORKSPACE = Path('/workspace')
SRC_DIR = WORKSPACE / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from parse.data_loading import load_base_records, load_grouped_unknown_records, load_ver10_seed_predictions
from parse.features import text_similarity
from parse.models import predict_ver10_improved

DATA_DIR = WORKSPACE / 'data'
NOTEBOOK_DIR = WORKSPACE / 'notebook'
OUTPUT_DIR = NOTEBOOK_DIR / 'ver13_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = DATA_DIR / 'annotations.jsonl'
GT_PATH = DATA_DIR / 'annotations_gt_task_ver09.json'
GROUPED_PATHS = [DATA_DIR / 'annotations_grouped_ver01.json', DATA_DIR / 'annotations_grouped_ver02.json']
VER10_SEED_PATH = NOTEBOOK_DIR / 'prediction_results_ver10.json'

print({'output_dir': str(OUTPUT_DIR)})

{'output_dir': '/workspace/notebook/ver13_outputs'}


## 1. データ読み込み
- 既知データで試行錯誤し、最終的に未知instructionにも適用する。

In [2]:
base_records = load_base_records(RAW_PATH, GT_PATH)
seed_predictions = load_ver10_seed_predictions(VER10_SEED_PATH)
unknown_records = load_grouped_unknown_records(GROUPED_PATHS, base_records, instruction_keys=('ver2', 'ver3', 'ver4'))

print('base_records:', len(base_records))
print('unknown_records:', len(unknown_records))
print('sample gt task count:', len(base_records[0]['gt_tasks']))
pprint(base_records[0]['gt_tasks'])

base_records: 100
unknown_records: 600
sample gt task count: 2
[{'action': 'dolly_in',
  'constraints': ['smooth_motion'],
  'params': {'end_framing': 'close_up',
             'motion_type': 'dolly_in',
             'start_framing': 'medium_shot'},
  'target': "man's face"},
 {'action': 'preserve_framing',
  'constraints': [],
  'params': {'position': 'center'},
  'target': "man's face"}]


## 2. multi task 評価関数
- GT task 群と予測 task 群の両方を評価する。
- 指標: coverage(再現側), precision(適合側), count_alignment(個数整合)

In [3]:
def normalize_text(x):
    if x is None:
        return ''
    t = str(x).lower().replace('_', ' ').strip()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def flatten_json(v, prefix=''):
    out = {}
    if isinstance(v, dict):
        for k, c in v.items():
            p = f'{prefix}.{k}' if prefix else str(k)
            out.update(flatten_json(c, p))
    elif isinstance(v, list):
        for i, c in enumerate(v):
            p = f'{prefix}[{i}]'
            out.update(flatten_json(c, p))
    else:
        out[prefix] = normalize_text(v)
    return out

def task_score(pred_task, gt_task):
    pa = pred_task.get('action', '')
    ga = gt_task.get('action', '')
    action = 1.0 if pa == ga else 0.0

    pt = pred_task.get('target', '')
    gt = gt_task.get('target', '')
    if isinstance(pt, list):
        pt = ' '.join(normalize_text(x) for x in pt)
    else:
        pt = normalize_text(pt)
    if isinstance(gt, list):
        gt = ' '.join(normalize_text(x) for x in gt)
    else:
        gt = normalize_text(gt)
    target = 0.0
    if pt and gt and (pt in gt or gt in pt):
        target = 1.0

    pp = flatten_json(pred_task.get('params', {}))
    gp = flatten_json(gt_task.get('params', {}))
    if not gp:
        params = 1.0
    elif not pp:
        params = 0.0
    else:
        m = 0
        for k, gv in gp.items():
            pv = pp.get(k, '')
            if pv and (pv == gv or pv in gv or gv in pv):
                m += 1
        params = m / len(gp)

    total = 0.5 * action + 0.2 * target + 0.3 * params
    return {'action': action, 'target': target, 'params': params, 'total': total}

def multi_task_score(pred_tasks, gt_tasks):
    pred_tasks = pred_tasks or []
    gt_tasks = gt_tasks or []

    if not pred_tasks and not gt_tasks:
        return {'coverage': 1.0, 'precision': 1.0, 'count_alignment': 1.0, 'total': 1.0}

    if gt_tasks:
        cov_vals = []
        for gt in gt_tasks:
            best = 0.0
            for pred in pred_tasks:
                best = max(best, task_score(pred, gt)['total'])
            cov_vals.append(best)
        coverage = sum(cov_vals) / len(cov_vals)
    else:
        coverage = 0.0

    if pred_tasks:
        pre_vals = []
        for pred in pred_tasks:
            best = 0.0
            for gt in gt_tasks:
                best = max(best, task_score(pred, gt)['total'])
            pre_vals.append(best)
        precision = sum(pre_vals) / len(pre_vals)
    else:
        precision = 0.0

    m = max(1, len(pred_tasks), len(gt_tasks))
    count_alignment = 1.0 - abs(len(pred_tasks) - len(gt_tasks)) / m

    total = 0.55 * coverage + 0.35 * precision + 0.10 * count_alignment
    return {
        'coverage': round(coverage, 4),
        'precision': round(precision, 4),
        'count_alignment': round(count_alignment, 4),
        'total': round(total, 4),
    }

print('multi task scorer ready')

multi task scorer ready


## 3. 試行 v13a: single-task をそのまま利用 (基準線)

In [4]:
def predict_v13a(record):
    pred, dbg = predict_ver10_improved(record, base_records)
    return pred, {'version': 'v13a_single_to_multi', **dbg}

def evaluate_model(records, predict_fn):
    rows = []
    predictions = {}
    debug_map = {}
    for r in records:
        pred, dbg = predict_fn(r)
        predictions[r['prediction_key']] = pred
        debug_map[r['prediction_key']] = dbg
        s = multi_task_score(pred.get('tasks', []), r['gt_tasks'])
        rows.append({
            'prediction_key': r['prediction_key'],
            'video_path': r['video_path'],
            'task_count_pred': len(pred.get('tasks', [])),
            'task_count_gt': len(r['gt_tasks']),
            **s,
        })

    overall = {k: round(sum(x[k] for x in rows) / len(rows), 4) for k in ['coverage', 'precision', 'count_alignment', 'total']}
    return {'rows': rows, 'overall': overall, 'predictions': predictions, 'debug': debug_map}

res_v13a = evaluate_model(base_records, predict_v13a)
print('v13a overall:', res_v13a['overall'])

v13a overall: {'coverage': 0.5746, 'precision': 0.8591, 'count_alignment': 0.443, 'total': 0.661}


## 4. 試行 v13b: preserve/stabilize 補助taskを追加
- 指示文の保護系表現 (`keep`, `preserve`, `maintain`, `stable`, `flicker`) を第2 taskとして追加する。

In [5]:
def infer_aux_tasks(record):
    t = record['instruction'].lower()
    aux = []

    if any(k in t for k in ['preserve identity', 'keep identity', 'identity']) and ('change' in t or 'replace' in t or 'edit' in t):
        aux.append({'action': 'preserve_identity', 'target': 'subject_identity', 'constraints': [], 'params': {}})

    if any(k in t for k in ['no flicker', 'without flicker', 'flickering', 'temporally consistent', 'stable', 'consistency']):
        aux.append({'action': 'stabilize_edit', 'target': 'full_frame', 'constraints': [], 'params': {}})

    if any(k in t for k in ['keep', 'preserve', 'maintain']) and any(k in t for k in ['background', 'layout', 'framing']):
        aux.append({'action': 'preserve_layout', 'target': 'scene_layout', 'constraints': [], 'params': {}})

    unique = []
    seen = set()
    for a in aux:
        key = (a['action'], a['target'])
        if key in seen:
            continue
        seen.add(key)
        unique.append(a)
    return unique

def predict_v13b(record):
    base_pred, base_dbg = predict_v13a(record)
    tasks = copy.deepcopy(base_pred.get('tasks', []))
    tasks.extend(infer_aux_tasks(record))
    return {'tasks': tasks}, {
        'version': 'v13b_rule_plus_aux',
        'base_confidence': base_dbg.get('confidence', 0.0),
        'aux_count': max(0, len(tasks) - 1),
    }

res_v13b = evaluate_model(base_records, predict_v13b)
print('v13b overall:', res_v13b['overall'])

v13b overall: {'coverage': 0.5902, 'precision': 0.6804, 'count_alignment': 0.6832, 'total': 0.6311}


## 5. 試行 v13c: 類似例の GT task 列を転写
- 最も近い既知instructionの GT task 個数・副次taskを転写し、先頭taskだけ現instructionの推定に差し替える。

In [6]:
def nearest_record(record, pool):
    best = None
    best_s = -1.0
    for c in pool:
        if c['video_path'] == record['video_path'] and c.get('variant') == record.get('variant'):
            continue
        s = text_similarity(record['instruction'], c['instruction'])
        if c['class'] == record['class']:
            s += 0.1
        if c['subclass'] == record['subclass']:
            s += 0.1
        if s > best_s:
            best_s = s
            best = c
    return best, best_s

def predict_v13c(record):
    p1, d1 = predict_v13a(record)
    nrec, ns = nearest_record(record, base_records)
    tasks = copy.deepcopy(p1.get('tasks', []))

    if nrec is not None:
        for gt_task in nrec.get('gt_tasks', [])[1:3]:
            tasks.append(copy.deepcopy(gt_task))

    unique = []
    seen = set()
    for t in tasks:
        key = (t.get('action', ''), json.dumps(t.get('target', ''), ensure_ascii=False, sort_keys=True))
        if key in seen:
            continue
        seen.add(key)
        unique.append(t)

    return {'tasks': unique}, {
        'version': 'v13c_retrieval_transfer',
        'nearest_score': round(float(ns), 4) if nrec is not None else 0.0,
        'nearest_video': nrec['video_path'] if nrec is not None else None,
    }

res_v13c = evaluate_model(base_records, predict_v13c)
print('v13c overall:', res_v13c['overall'])

v13c overall: {'coverage': 0.7029, 'precision': 0.8163, 'count_alignment': 0.6703, 'total': 0.7393}


## 6. 試行 v13d: 簡易アンサンブル
- v13b と v13c のうち、instruction の保護系語彙密度と retrieval 類似度で動的選択する。

In [7]:
def preservation_density(text):
    t = text.lower()
    keys = ['preserve', 'maintain', 'keep', 'consistent', 'stable', 'flicker', 'artifact']
    cnt = sum(1 for k in keys if k in t)
    return cnt / max(1, len(keys))

def predict_v13d(record):
    pb, db = predict_v13b(record)
    pc, dc = predict_v13c(record)

    pden = preservation_density(record['instruction'])
    nscore = dc.get('nearest_score', 0.0)

    # preserve語彙が多い時は v13b を優先、類似例が非常に高い時は v13c を優先
    use_c = nscore >= 0.92 and pden < 0.35
    chosen = pc if use_c else pb
    source = 'v13c' if use_c else 'v13b'

    return chosen, {
        'version': 'v13d_ensemble',
        'selected_from': source,
        'preservation_density': round(pden, 4),
        'nearest_score': round(nscore, 4),
    }

res_v13d = evaluate_model(base_records, predict_v13d)
print('v13d overall:', res_v13d['overall'])

summary = {
    'v13a_single_to_multi': res_v13a['overall'],
    'v13b_rule_plus_aux': res_v13b['overall'],
    'v13c_retrieval_transfer': res_v13c['overall'],
    'v13d_ensemble': res_v13d['overall'],
}
print('\nbase-record benchmark')
for k, v in sorted(summary.items(), key=lambda kv: kv[1]['total'], reverse=True):
    print(k, v)

v13d overall: {'coverage': 0.5902, 'precision': 0.6804, 'count_alignment': 0.6832, 'total': 0.6311}

base-record benchmark
v13c_retrieval_transfer {'coverage': 0.7029, 'precision': 0.8163, 'count_alignment': 0.6703, 'total': 0.7393}
v13a_single_to_multi {'coverage': 0.5746, 'precision': 0.8591, 'count_alignment': 0.443, 'total': 0.661}
v13b_rule_plus_aux {'coverage': 0.5902, 'precision': 0.6804, 'count_alignment': 0.6832, 'total': 0.6311}
v13d_ensemble {'coverage': 0.5902, 'precision': 0.6804, 'count_alignment': 0.6832, 'total': 0.6311}


## 7. 未知instructionへ適用して保存
- base_records で最良だった方式を unknown_records に適用する。

In [8]:
model_results = {
    'v13a_single_to_multi': evaluate_model(unknown_records, predict_v13a),
    'v13b_rule_plus_aux': evaluate_model(unknown_records, predict_v13b),
    'v13c_retrieval_transfer': evaluate_model(unknown_records, predict_v13c),
    'v13d_ensemble': evaluate_model(unknown_records, predict_v13d),
}

unknown_summary = {k: v['overall'] for k, v in model_results.items()}
best_name = max(unknown_summary, key=lambda n: unknown_summary[n]['total'])

print('unknown-instruction benchmark')
for k, v in sorted(unknown_summary.items(), key=lambda kv: kv[1]['total'], reverse=True):
    print(k, v)

print('best model:', best_name)

for name, result in model_results.items():
    rows = []
    pred_map = result['predictions']
    dbg_map = result['debug']
    score_by_key = {r['prediction_key']: r for r in result['rows']}
    for rec in unknown_records:
        key = rec['prediction_key']
        rows.append({
            'prediction_key': key,
            'video_path': rec['video_path'],
            'variant': rec.get('variant', 'base'),
            'variant_source': rec.get('variant_source', 'base'),
            'instruction': rec['instruction'],
            'class': rec['class'],
            'subclass': rec['subclass'],
            'gt_tasks': rec['gt_tasks'],
            'prediction': pred_map[key],
            'debug': dbg_map[key],
            'scores': {
                'coverage': score_by_key[key]['coverage'],
                'precision': score_by_key[key]['precision'],
                'count_alignment': score_by_key[key]['count_alignment'],
                'total': score_by_key[key]['total'],
            },
        })

    out_path = OUTPUT_DIR / f'unknown_predictions_{name}.json'
    out_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')

analysis = {
    'known_benchmark': summary,
    'unknown_benchmark': unknown_summary,
    'best_model': best_name,
    'record_count': len(unknown_records),
    'note': 'This notebook experiments with multi-task prediction after primary-task versions.',
}
analysis_path = OUTPUT_DIR / 'multi_task_analysis_ver13.json'
analysis_path.write_text(json.dumps(analysis, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved:', analysis_path)

unknown-instruction benchmark
v13c_retrieval_transfer {'coverage': 0.8533, 'precision': 0.9442, 'count_alignment': 0.8323, 'total': 0.883}
v13d_ensemble {'coverage': 0.712, 'precision': 0.7692, 'count_alignment': 0.7938, 'total': 0.7402}
v13a_single_to_multi {'coverage': 0.5929, 'precision': 0.8913, 'count_alignment': 0.443, 'total': 0.6823}
v13b_rule_plus_aux {'coverage': 0.6096, 'precision': 0.6532, 'count_alignment': 0.6718, 'total': 0.6311}
best model: v13c_retrieval_transfer
saved: /workspace/notebook/ver13_outputs/multi_task_analysis_ver13.json


## 8. 誤差分析（best model）

In [9]:
best_rows = sorted(model_results[best_name]['rows'], key=lambda x: x['total'])
print('worst 5 examples:', best_name)
for row in best_rows[:5]:
    key = row['prediction_key']
    rec = next(r for r in unknown_records if r['prediction_key'] == key)
    pred = model_results[best_name]['predictions'][key]
    print('=' * 100)
    print('prediction_key:', key)
    print('instruction:', rec['instruction'])
    print('gt_task_count:', len(rec['gt_tasks']))
    print('pred_task_count:', len(pred.get('tasks', [])))
    print('score:', {k: row[k] for k in ['coverage', 'precision', 'count_alignment', 'total']})
    print('gt_tasks:')
    pprint(rec['gt_tasks'])
    print('pred_tasks:')
    pprint(pred.get('tasks', []))

worst 5 examples: v13c_retrieval_transfer
prediction_key: MPCQi8LEeOI_67_0to214.mp4::annotations_grouped_ver01:ver2
instruction: Make the black radiator fans visible behind the metallic plate spin rapidly and continuously. Ensure consistency.
gt_task_count: 1
pred_task_count: 1
score: {'coverage': 0.5, 'precision': 0.5, 'count_alignment': 1.0, 'total': 0.55}
gt_tasks:
[{'action': 'edit_motion',
  'constraints': [],
  'params': {'gesture': 'spin'},
  'target': 'object'}]
pred_tasks:
[{'action': 'edit_motion',
  'constraints': [],
  'params': {'motion': 'spin'},
  'target': 'person'}]
prediction_key: MPCQi8LEeOI_67_0to214.mp4::annotations_grouped_ver01:ver3
instruction: Make the black radiator fans visible behind the metallic plate spin rapidly and continuously. Keep everything consistent.
gt_task_count: 1
pred_task_count: 1
score: {'coverage': 0.5, 'precision': 0.5, 'count_alignment': 1.0, 'total': 0.55}
gt_tasks:
[{'action': 'edit_motion',
  'constraints': [],
  'params': {'gesture': '